# 29. Export app search runtime snapshot

This notebook exports a deterministic, application-owned vector snapshot. It fingerprints the source tree before copying it to a temporary directory, opens **only that copy** with Chroma, and fails if the original source tree changes before extraction is complete. No network or embedding API calls are made.

In [1]:
from __future__ import annotations

import ast
import hashlib
import json
import os
import shutil
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import chromadb
import numpy as np

PROJECT_ROOT = next((candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (candidate / 'AGENTS.md').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from the repository or a subdirectory.')

SOURCE_CHROMA_ROOT = PROJECT_ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval/chroma'
SOURCE_QUERY_NOTEBOOK = PROJECT_ROOT / 'notebooks/13_hierarchical_chunking_retrieval.ipynb'
SNAPSHOT_ROOT = PROJECT_ROOT / 'data/search/benefit_hierarchy/v1'
DEVELOPMENT_ROOT = PROJECT_ROOT / 'data/development/card_search_v1'

SNAPSHOT_SCHEMA_VERSION = 'benefit_hierarchy_snapshot_v1'
DEVELOPMENT_SCHEMA_VERSION = 'card_search_development_v1'
EMBEDDING_MODEL = 'text-embedding-3-small'
DISTANCE_CONTRACT = 'squared_l2'
EXPECTED_CHUNK_COUNT = 327
EXPECTED_CARD_COUNT = 10
EXPECTED_EMBEDDING_DIMENSION = 1536
METADATA_ALLOWLIST = {
    'card_key', 'issuer', 'card_name', 'level', 'parent_id',
    'page_num', 'part_num', 'section',
}
FORBIDDEN_METADATA_TERMS = ('query', 'gold', 'expected', 'relevance', 'holdout')


def canonical_json(value: object) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def tree_state(root: Path) -> dict[str, object]:
    if not root.is_dir():
        raise FileNotFoundError(root)
    files = []
    for path in sorted(root.rglob('*')):
        if path.is_symlink() or not path.is_file():
            if path.is_symlink():
                raise RuntimeError(f'Source tree contains a symlink: {path}')
            continue
        stat = path.stat()
        files.append({
            'ordinal': len(files),
            'size_bytes': stat.st_size,
            'mtime_ns': stat.st_mtime_ns,
            'sha256': sha256_file(path),
        })
    if not files:
        raise RuntimeError('Source tree has no files.')
    return {'tree_sha256': sha256_bytes(canonical_json(files).encode('utf-8')), 'files': files}


def atomic_write_bytes(path: Path, value: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.tmp')
    with temporary.open('wb') as handle:
        handle.write(value)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def atomic_write_json(path: Path, value: object) -> None:
    atomic_write_bytes(path, (canonical_json(value) + '\n').encode('utf-8'))


def atomic_write_jsonl(path: Path, values: list[object]) -> None:
    atomic_write_bytes(path, ''.join(canonical_json(value) + '\n' for value in values).encode('utf-8'))


def contains_forbidden_metadata_term(value: object) -> bool:
    if isinstance(value, dict):
        return any(contains_forbidden_metadata_term(key) or contains_forbidden_metadata_term(item) for key, item in value.items())
    if isinstance(value, (list, tuple)):
        return any(contains_forbidden_metadata_term(item) for item in value)
    if isinstance(value, str):
        lowered = value.casefold()
        return any(term in lowered for term in FORBIDDEN_METADATA_TERMS)
    return False


def public_metadata(metadata: dict[str, object]) -> dict[str, object]:
    # The source metadata includes provenance and evaluation-only fields.
    # Keep only runtime fields, and omit a value if it encodes a protected term.
    result = {
        key: metadata[key] for key in sorted(METADATA_ALLOWLIST & set(metadata))
        if not contains_forbidden_metadata_term(metadata[key])
    }
    if set(result) - METADATA_ALLOWLIST or contains_forbidden_metadata_term(result):
        raise RuntimeError(f'Forbidden snapshot metadata: {result}')
    return result


def load_queries_and_gold(notebook_path: Path) -> list[dict[str, object]]:
    notebook = json.loads(notebook_path.read_text(encoding='utf-8'))
    for cell in notebook['cells']:
        source = ''.join(cell.get('source', []))
        if 'QUERIES = [' not in source:
            continue
        tree = ast.parse(source)
        for node in tree.body:
            if isinstance(node, ast.Assign) and any(isinstance(target, ast.Name) and target.id == 'QUERIES' for target in node.targets):
                raw_queries = ast.literal_eval(node.value)
                rows = []
                for item in raw_queries:
                    row = {
                        'id': item['id'],
                        'category': item['category'],
                        'text': item['query'],
                        'gold': {
                            'card_key': item['expected_card'],
                            'level': item['expected_level'],
                            'required_terms': item['required_terms'],
                        },
                    }
                    if item['id'] == 'ibk_semantic':
                        row['known_limitations'] = [
                            'Other cards may also answer this semantic query; the singleton gold label may be incomplete or ambiguous.'
                        ]
                    rows.append(row)
                if len(rows) != 30 or len({row['id'] for row in rows}) != 30:
                    raise RuntimeError('Expected 30 unique development query IDs.')
                return sorted(rows, key=lambda row: str(row['id']))
    raise RuntimeError('QUERIES assignment was not found in notebook 13.')


source_before = tree_state(SOURCE_CHROMA_ROOT)
extraction_error: Exception | None = None
records: list[dict[str, object]] = []
try:
    with tempfile.TemporaryDirectory(prefix='pickcardu_chroma_snapshot_', dir='/tmp') as temporary_dir:
        copied_root = Path(temporary_dir) / 'source_copy'
        shutil.copytree(SOURCE_CHROMA_ROOT, copied_root, copy_function=shutil.copy2)
        if tree_state(copied_root) != source_before:
            raise RuntimeError('Temporary source copy does not match the pre-export source fingerprint.')
        # Do not open SOURCE_CHROMA_ROOT with Chroma: extraction is from this isolated copy only.
        copied_client = chromadb.PersistentClient(path=str(copied_root))
        matching_collections = [collection for collection in copied_client.list_collections() if collection.count() == EXPECTED_CHUNK_COUNT]
        if len(matching_collections) != 1:
            raise RuntimeError(f'Expected exactly one {EXPECTED_CHUNK_COUNT}-chunk collection, got {len(matching_collections)}.')
        collection = matching_collections[0]
        extracted = collection.get(include=['documents', 'metadatas', 'embeddings'])
        raw_ids = extracted['ids']
        raw_documents = extracted['documents']
        raw_metadatas = extracted['metadatas']
        raw_embeddings = np.asarray(extracted['embeddings'])
        if not (len(raw_ids) == len(raw_documents) == len(raw_metadatas) == len(raw_embeddings) == EXPECTED_CHUNK_COUNT):
            raise RuntimeError('Chroma returned mismatched record lengths.')
        if len(set(raw_ids)) != EXPECTED_CHUNK_COUNT:
            raise RuntimeError('Chroma returned non-unique chunk IDs.')
        source_by_id = {
            chunk_id: (document, metadata, raw_embeddings[index])
            for index, (chunk_id, document, metadata) in enumerate(zip(raw_ids, raw_documents, raw_metadatas, strict=True))
        }
        sorted_ids = sorted(source_by_id)
        records = [
            {'id': chunk_id, 'document': source_by_id[chunk_id][0], 'metadata': public_metadata(source_by_id[chunk_id][1])}
            for chunk_id in sorted_ids
        ]
        embeddings = np.asarray([source_by_id[chunk_id][2] for chunk_id in sorted_ids], dtype=np.float32)
        # Validate that each sorted ID still owns its original document, metadata, and embedding.
        assert [record['id'] for record in records] == sorted_ids
        assert all(record['document'] == source_by_id[record['id']][0] for record in records)
        assert all(record['metadata'] == public_metadata(source_by_id[record['id']][1]) for record in records)
        assert all(np.array_equal(embeddings[index], np.asarray(source_by_id[record['id']][2], dtype=np.float32)) for index, record in enumerate(records))
except Exception as error:
    extraction_error = error
finally:
    source_after = tree_state(SOURCE_CHROMA_ROOT)

if source_before != source_after:
    raise RuntimeError('Source tree changed during export; no snapshot was written.')
if extraction_error is not None:
    raise extraction_error

if len(records) != EXPECTED_CHUNK_COUNT or len({record['id'] for record in records}) != EXPECTED_CHUNK_COUNT:
    raise RuntimeError('Snapshot chunk-count contract failed.')
if embeddings.ndim != 2 or embeddings.shape != (EXPECTED_CHUNK_COUNT, EXPECTED_EMBEDDING_DIMENSION):
    raise RuntimeError(f'Embedding shape contract failed: {embeddings.shape}')
if not np.issubdtype(embeddings.dtype, np.floating) or not np.isfinite(embeddings).all():
    raise RuntimeError('Embedding dtype or finite-value contract failed.')

card_rows = sorted(
    [
        {
            'id': record['id'],
            'card_key': record['metadata']['card_key'],
            'issuer': record['metadata']['issuer'],
            'card_name': record['metadata']['card_name'],
        }
        for record in records if record['metadata'].get('level') == 'card'
    ],
    key=lambda row: (str(row['card_key']), str(row['id'])),
)
if len(card_rows) != EXPECTED_CARD_COUNT or len({row['card_key'] for row in card_rows}) != EXPECTED_CARD_COUNT:
    raise RuntimeError('Snapshot card-count contract failed.')

created_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z')
snapshot_id = sha256_bytes(canonical_json({
    'schema_version': SNAPSHOT_SCHEMA_VERSION,
    'source_tree_sha256': source_before['tree_sha256'],
    'embedding_model': EMBEDDING_MODEL,
    'distance_contract': DISTANCE_CONTRACT,
}).encode('utf-8'))[:24]

SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)
atomic_write_json(SNAPSHOT_ROOT / 'cards.json', card_rows)
atomic_write_jsonl(SNAPSHOT_ROOT / 'chunks.jsonl', records)
temporary_embeddings = SNAPSHOT_ROOT / '.embeddings.npy.tmp'
with temporary_embeddings.open('wb') as handle:
    np.save(handle, embeddings, allow_pickle=False)
    handle.flush()
    os.fsync(handle.fileno())
os.replace(temporary_embeddings, SNAPSHOT_ROOT / 'embeddings.npy')

snapshot_checksums = {
    'cards.json': sha256_file(SNAPSHOT_ROOT / 'cards.json'),
    'chunks.jsonl': sha256_file(SNAPSHOT_ROOT / 'chunks.jsonl'),
    'embeddings.npy': sha256_file(SNAPSHOT_ROOT / 'embeddings.npy'),
}
snapshot_manifest = {
    'schema_version': SNAPSHOT_SCHEMA_VERSION,
    'snapshot_id': snapshot_id,
    'created_at': created_at,
    'embedding_model': EMBEDDING_MODEL,
    'distance_contract': DISTANCE_CONTRACT,
    'source_fingerprint': source_before,
    'source_verification_after_export': source_after,
    'records': {'chunks': EXPECTED_CHUNK_COUNT, 'unique_chunk_ids': EXPECTED_CHUNK_COUNT, 'cards': EXPECTED_CARD_COUNT, 'unique_card_keys': EXPECTED_CARD_COUNT},
    'embeddings': {'shape': list(embeddings.shape), 'dtype': str(embeddings.dtype), 'finite': True},
    'metadata_contract': {'allowed_fields': sorted(METADATA_ALLOWLIST), 'forbidden_terms': list(FORBIDDEN_METADATA_TERMS)},
    'artifact_sha256': snapshot_checksums,
    'manifest_sha256': None,
    'manifest_sha256_scope': 'SHA-256 of canonical manifest JSON with manifest_sha256 set to null; avoids an impossible self-referential file checksum.',
}
snapshot_manifest['manifest_sha256'] = sha256_bytes((canonical_json(snapshot_manifest) + '\n').encode('utf-8'))
atomic_write_json(SNAPSHOT_ROOT / 'manifest.json', snapshot_manifest)

development_rows = load_queries_and_gold(SOURCE_QUERY_NOTEBOOK)
DEVELOPMENT_ROOT.mkdir(parents=True, exist_ok=True)
atomic_write_jsonl(DEVELOPMENT_ROOT / 'queries.jsonl', development_rows)
snapshot_binding = {
    'snapshot_id': snapshot_manifest['snapshot_id'],
    'canonical_manifest_sha256': snapshot_manifest['manifest_sha256'],
    'artifact_sha256': snapshot_manifest['artifact_sha256'],
}
development_manifest = {
    'schema_version': DEVELOPMENT_SCHEMA_VERSION,
    'created_at': created_at,
    'purpose': 'development retrieval diagnostics; not a product success criterion.',
    'query_count': len(development_rows),
    'query_ids_sorted': [row['id'] for row in development_rows],
    'source_notebook_sha256': sha256_file(SOURCE_QUERY_NOTEBOOK),
    'snapshot_binding': snapshot_binding,
    'metric_contract': {
        'gold_cardinality': 'singleton',
        'hit_at_k_equals_recall_at_k': True,
        'explanation': 'Each query currently has one gold relevance target, so Hit@K and Recall@K are equal for this diagnostic dataset.',
    },
    'label_limitations': {
        'semantic_relevance_may_be_incomplete_or_ambiguous': True,
        'explanation': 'Some semantic-query relevance labels may be incomplete or ambiguous and must not be treated as exhaustive relevance judgments.',
        'query_level': {
            'ibk_semantic': 'Other cards may also answer this semantic query; its singleton gold label is not exhaustive.'
        },
    },
    'usage_contract': {
        'development_diagnostics_only': True,
        'product_success_criterion': False,
        'product_performance_decision_allowed': False,
        'generalization_or_promotion_decision_allowed': False,
        'explanation': 'This dataset supports development retrieval diagnostics only; it cannot establish product performance, generalization, or promotion readiness.',
    },
    'query_embedding_fixture': {'status': 'pending_external_approval', 'reason': 'No approved OpenAI embedding call; no query embeddings were generated.'},
    'comparison_readiness': {
        'fixed_development_comparison_allowed': False,
        'blocked_by': 'query_embedding_fixture.pending_external_approval',
        'requirement_to_unblock': 'Approved frozen query embedding fixture matching the bound snapshot embedding model and distance contract.',
    },
    'artifact_sha256': {'queries.jsonl': sha256_file(DEVELOPMENT_ROOT / 'queries.jsonl')},
    'manifest_sha256': None,
    'manifest_sha256_scope': 'SHA-256 of canonical manifest JSON with manifest_sha256 set to null; avoids an impossible self-referential file checksum.',
}
development_manifest['manifest_sha256'] = sha256_bytes((canonical_json(development_manifest) + '\n').encode('utf-8'))
atomic_write_json(DEVELOPMENT_ROOT / 'manifest.json', development_manifest)

print({
    'snapshot_id': snapshot_id,
    'source_tree_sha256': source_before['tree_sha256'],
    'source_unchanged_after_export': source_before == source_after,
    'chunks': len(records),
    'cards': len(card_rows),
    'embedding_shape': list(embeddings.shape),
    'embedding_dtype': str(embeddings.dtype),
    'snapshot_checksums': snapshot_checksums,
    'development_queries': len(development_rows),
})


{'snapshot_id': '7a2f886b257e79b5550b0192', 'source_tree_sha256': '9fa856006cd5edadc46d4a22fd243ccb059106e4d3d011a14834b426d7f5a525', 'source_unchanged_after_export': True, 'chunks': 327, 'cards': 10, 'embedding_shape': [327, 1536], 'embedding_dtype': 'float32', 'snapshot_checksums': {'cards.json': '3c152ac6ea5f3c3547e309536504d2c9c008c0a1c86ee531b6b4746e79b64b87', 'chunks.jsonl': 'e9c94f81c6eac59eb821a1f4b7c95da882d1de1648cc64e0d17ef760251f7106', 'embeddings.npy': 'e55ce9a74c2a21de6ac4910888d0339199e22f645ca840b480deeac6d2068958'}, 'development_queries': 30}
